# SQL Analysis: Player Subscription & Engagement Queries

**Purpose:** Demonstrate SQL proficiency through 12+ queries covering JOINs, window functions, CTEs, subqueries, and cohort analysis. These queries mirror the day-to-day work of extracting insights from player behavioural data.

**Approach:** We use SQLite (via Python's `sqlite3`) to query the same 5 CSV tables used throughout this project. Each query includes:
- The business question it answers
- SQL query with comments
- PySpark translation (documented in comments)
- Result interpretation

**Tables:** `accounts`, `subscriptions`, `feature_usage`, `support_tickets`, `churn_events`

In [1]:
import sqlite3
import sys
from pathlib import Path

import pandas as pd
import yaml

# Project paths
PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

with open(PROJECT_ROOT / "config.yaml", "r") as f:
    config = yaml.safe_load(f)

# Load raw data into SQLite in-memory database
conn = sqlite3.connect(":memory:")

tables = {
    "accounts": PROJECT_ROOT / config["data"]["raw_files"]["accounts"],
    "subscriptions": PROJECT_ROOT / config["data"]["raw_files"]["subscriptions"],
    "feature_usage": PROJECT_ROOT / config["data"]["raw_files"]["feature_usage"],
    "support_tickets": PROJECT_ROOT / config["data"]["raw_files"]["support_tickets"],
    "churn_events": PROJECT_ROOT / config["data"]["raw_files"]["churn_events"],
}

for name, path in tables.items():
    df = pd.read_csv(path)
    df.to_sql(name, conn, index=False, if_exists="replace")
    print(f"Loaded {name}: {len(df)} rows")

def query(sql, conn=conn):
    """Execute SQL and return as DataFrame."""
    return pd.read_sql_query(sql, conn)

Loaded accounts: 500 rows
Loaded subscriptions: 5000 rows
Loaded feature_usage: 25000 rows
Loaded support_tickets: 2000 rows
Loaded churn_events: 600 rows


---
## 1. Player Overview: Multi-Table JOIN

**Business Question:** What is the complete profile of each player -- account details, subscription count, total revenue, and support interactions?

**SQL Skills:** LEFT JOIN across 3 tables, aggregation, COALESCE for null handling

In [2]:
sql_1 = """
-- Player 360-degree profile: account + subscriptions + support
-- PySpark: accounts.join(subs_agg, 'account_id', 'left').join(tickets_agg, 'account_id', 'left')

SELECT
    a.account_id,
    a.signup_date,
    a.plan_tier,
    a.referral_source,
    a.churn_flag,
    COALESCE(s.subscription_count, 0)       AS subscription_count,
    COALESCE(s.total_mrr, 0)                AS total_mrr,
    COALESCE(s.avg_mrr, 0)                  AS avg_mrr,
    COALESCE(t.ticket_count, 0)             AS ticket_count,
    COALESCE(t.avg_satisfaction, 0)          AS avg_satisfaction
FROM accounts a
LEFT JOIN (
    SELECT
        account_id,
        COUNT(*)                AS subscription_count,
        SUM(mrr_amount)         AS total_mrr,
        AVG(mrr_amount)         AS avg_mrr
    FROM subscriptions
    GROUP BY account_id
) s ON a.account_id = s.account_id
LEFT JOIN (
    SELECT
        account_id,
        COUNT(*)                            AS ticket_count,
        AVG(satisfaction_score)              AS avg_satisfaction
    FROM support_tickets
    GROUP BY account_id
) t ON a.account_id = t.account_id
ORDER BY total_mrr DESC
LIMIT 10;
"""

result_1 = query(sql_1)
print(f"Top 10 players by total revenue:")
result_1

Top 10 players by total revenue:


,account_id,signup_date,plan_tier,referral_source,churn_flag,subscription_count,total_mrr,avg_mrr,ticket_count,avg_satisfaction
0,A-d4e0d4,2024-10-18,Basic,other,0,10,138060,13806.000000,4,4.0
1,A-5b1bcd,2023-09-16,Pro,organic,0,15,133298,8886.533333,4,4.5
2,A-5a215a,2024-01-05,Enterprise,other,0,18,130152,7230.666667,4,4.0
3,A-1f0636,2024-01-01,Pro,partner,0,11,94710,8610.000000,2,3.5
4,A-4814a3,2024-09-17,Basic,event,1,11,87957,7996.090909,4,4.0
5,A-5c046d,2024-07-06,Basic,partner,0,8,83886,10485.750000,1,3.0
6,A-18793f,2024-12-18,Basic,organic,0,11,78588,7144.363636,5,3.8
7,A-30b4ca,2024-05-20,Basic,organic,1,8,75938,9492.250000,3,4.0
8,A-e08cd3,2024-09-21,Pro,ads,0,11,71020,6456.363636,8,4.0
9,A-c58f49,2024-07-31,Enterprise,other,0,7,70980,10140.000000,3,5.0


---
## 2. Churn Rate by Acquisition Channel

**Business Question:** Which referral sources produce the highest-quality (lowest churn) players? Should the UA team shift budget allocation?

**SQL Skills:** GROUP BY with conditional aggregation (CASE WHEN), percentage calculation

In [3]:
sql_2 = """
-- Churn rate by referral source (acquisition channel)
-- PySpark: accounts.groupBy('referral_source').agg(
--     F.count('*').alias('total'),
--     F.sum(F.when(F.col('churn_flag'), 1).otherwise(0)).alias('churned')
-- ).withColumn('churn_rate', F.col('churned') / F.col('total'))

SELECT
    referral_source,
    COUNT(*)                                                AS total_players,
    SUM(CASE WHEN churn_flag = 1 THEN 1 ELSE 0 END)        AS churned_players,
    ROUND(100.0 * SUM(CASE WHEN churn_flag = 1 THEN 1 ELSE 0 END) / COUNT(*), 1)
                                                            AS churn_rate_pct,
    ROUND(AVG(seats), 1)                                    AS avg_seats
FROM accounts
GROUP BY referral_source
ORDER BY churn_rate_pct;
"""

result_2 = query(sql_2)
print("Churn rate by acquisition channel:")
result_2

Churn rate by acquisition channel:


,referral_source,total_players,churned_players,churn_rate_pct,avg_seats
0,partner,89,13,14.6,20.6
1,organic,114,20,17.5,20.2
2,ads,98,23,23.5,19.0
3,other,103,25,24.3,21.8
4,event,96,29,30.2,21.2


---
## 3. Revenue Ranking with Window Functions

**Business Question:** How does each player rank in revenue within their subscription tier? Who are the top 3 spenders per tier?

**SQL Skills:** RANK(), ROW_NUMBER(), PARTITION BY, window functions

In [4]:
sql_3 = """
-- Revenue ranking within each plan tier using window functions
-- PySpark: Window.partitionBy('plan_tier').orderBy(F.desc('total_mrr'))
--          df.withColumn('rank', F.rank().over(w))

WITH player_revenue AS (
    SELECT
        a.account_id,
        a.plan_tier,
        SUM(s.mrr_amount)   AS total_mrr,
        COUNT(s.subscription_id) AS sub_count
    FROM accounts a
    JOIN subscriptions s ON a.account_id = s.account_id
    GROUP BY a.account_id, a.plan_tier
),
ranked AS (
    SELECT
        *,
        RANK() OVER (PARTITION BY plan_tier ORDER BY total_mrr DESC)  AS revenue_rank,
        ROW_NUMBER() OVER (PARTITION BY plan_tier ORDER BY total_mrr DESC) AS row_num
    FROM player_revenue
)
SELECT
    plan_tier,
    account_id,
    total_mrr,
    sub_count,
    revenue_rank
FROM ranked
WHERE row_num <= 3
ORDER BY plan_tier, revenue_rank;
"""

result_3 = query(sql_3)
print("Top 3 revenue players per tier:")
result_3

Top 3 revenue players per tier:


,plan_tier,account_id,total_mrr,sub_count,revenue_rank
0,Basic,A-d4e0d4,138060,10,1
1,Basic,A-4814a3,87957,11,2
2,Basic,A-5c046d,83886,8,3
3,Enterprise,A-5a215a,130152,18,1
4,Enterprise,A-c58f49,70980,7,2
5,Enterprise,A-09316c,70000,11,3
6,Pro,A-5b1bcd,133298,15,1
7,Pro,A-1f0636,94710,11,2
8,Pro,A-e08cd3,71020,11,3


---
## 4. Cohort Retention Analysis with CTEs

**Business Question:** What is the Day-30 / Day-90 / Day-180 retention rate for each monthly signup cohort? Are newer cohorts retaining better?

**SQL Skills:** CTEs (Common Table Expressions), date arithmetic, cohort analysis, conditional aggregation

In [5]:
sql_4 = """
-- Cohort retention analysis: signup month cohorts with 30/90/180-day retention
-- PySpark: accounts.withColumn('cohort', F.date_trunc('month', 'signup_date'))
--          .join(churn_events, ...).groupBy('cohort').agg(...)

WITH cohorts AS (
    SELECT
        account_id,
        signup_date,
        churn_flag,
        SUBSTR(signup_date, 1, 7) AS signup_cohort,
        -- Calculate tenure: days from signup to churn or to analysis date
        CASE
            WHEN churn_flag = 1 THEN
                JULIANDAY(
                    COALESCE(
                        (SELECT MAX(c.churn_date) FROM churn_events c WHERE c.account_id = a.account_id),
                        '2024-12-31'
                    )
                ) - JULIANDAY(signup_date)
            ELSE
                JULIANDAY('2024-12-31') - JULIANDAY(signup_date)
        END AS tenure_days
    FROM accounts a
)
SELECT
    signup_cohort,
    COUNT(*)                                                    AS cohort_size,
    ROUND(100.0 * SUM(CASE WHEN tenure_days >= 30 THEN 1 ELSE 0 END) / COUNT(*), 1)
                                                                AS retention_30d_pct,
    ROUND(100.0 * SUM(CASE WHEN tenure_days >= 90 THEN 1 ELSE 0 END) / COUNT(*), 1)
                                                                AS retention_90d_pct,
    ROUND(100.0 * SUM(CASE WHEN tenure_days >= 180 THEN 1 ELSE 0 END) / COUNT(*), 1)
                                                                AS retention_180d_pct
FROM cohorts
GROUP BY signup_cohort
ORDER BY signup_cohort;
"""

result_4 = query(sql_4)
print("Cohort retention analysis:")
result_4

Cohort retention analysis:


,signup_cohort,cohort_size,retention_30d_pct,retention_90d_pct,retention_180d_pct
0,2023-01,17,100.0,100.0,94.1
1,2023-02,18,100.0,100.0,100.0
2,2023-03,20,100.0,95.0,95.0
3,2023-04,15,100.0,100.0,100.0
4,2023-05,26,100.0,100.0,100.0
5,2023-06,13,100.0,100.0,100.0
6,2023-07,14,100.0,100.0,100.0
7,2023-08,16,100.0,100.0,100.0
8,2023-09,23,100.0,100.0,95.7
9,2023-10,20,100.0,100.0,95.0


---
## 5. Running Revenue with LAG Window Function

**Business Question:** How does each player's monthly revenue compare to their previous month? Is their spending accelerating or decelerating?

**SQL Skills:** LAG() window function, month-over-month comparison, percentage change calculation

In [6]:
sql_5 = """
-- Month-over-month revenue trend per player using LAG()
-- PySpark: Window.partitionBy('account_id').orderBy('month')
--          df.withColumn('prev_rev', F.lag('monthly_mrr').over(w))

WITH monthly_revenue AS (
    SELECT
        account_id,
        SUBSTR(start_date, 1, 7)     AS revenue_month,
        SUM(mrr_amount)              AS monthly_mrr
    FROM subscriptions
    GROUP BY account_id, SUBSTR(start_date, 1, 7)
),
with_lag AS (
    SELECT
        *,
        LAG(monthly_mrr) OVER (PARTITION BY account_id ORDER BY revenue_month)
            AS prev_month_mrr,
        ROW_NUMBER() OVER (PARTITION BY account_id ORDER BY revenue_month DESC)
            AS recency_rank
    FROM monthly_revenue
)
SELECT
    account_id,
    revenue_month,
    ROUND(monthly_mrr, 2)       AS monthly_mrr,
    ROUND(prev_month_mrr, 2)    AS prev_month_mrr,
    ROUND(
        CASE
            WHEN prev_month_mrr > 0
            THEN 100.0 * (monthly_mrr - prev_month_mrr) / prev_month_mrr
            ELSE NULL
        END, 1
    ) AS mom_change_pct
FROM with_lag
WHERE recency_rank <= 3  -- last 3 months per player
ORDER BY account_id, revenue_month DESC
LIMIT 15;
"""

result_5 = query(sql_5)
print("Month-over-month revenue changes (last 3 months per player):")
result_5

Month-over-month revenue changes (last 3 months per player):


,account_id,revenue_month,monthly_mrr,prev_month_mrr,mom_change_pct
0,A-00bed1,2024-09,5572.0,3136.0,77.7
1,A-00bed1,2024-08,3136.0,6944.0,-54.8
2,A-00bed1,2024-07,6944.0,15323.0,-54.7
3,A-00cac8,2024-12,931.0,3980.0,-76.6
4,A-00cac8,2024-10,3980.0,392.0,915.3
5,A-00cac8,2024-09,392.0,7913.0,-95.0
6,A-0158bb,2024-12,3234.0,0.0,NaN
7,A-0158bb,2024-10,0.0,836.0,-100.0
8,A-0158bb,2024-06,836.0,0.0,NaN
9,A-016043,2024-12,3081.0,4985.0,-38.2


---
## 6. Churn Reason Analysis with Correlated Subquery

**Business Question:** What are the primary churn reasons, and how do they correlate with player revenue and tenure?

**SQL Skills:** Correlated subquery, GROUP BY with multiple aggregations, HAVING filter

In [7]:
sql_6 = """
-- Churn reasons enriched with player revenue and tenure
-- PySpark: churn_events.join(accounts, 'account_id')
--          .join(subs_agg, 'account_id').groupBy('reason_code').agg(...)

SELECT
    ce.reason_code,
    COUNT(DISTINCT ce.account_id)       AS unique_players,
    ROUND(AVG(
        (SELECT SUM(s.mrr_amount)
         FROM subscriptions s
         WHERE s.account_id = ce.account_id)
    ), 2)                               AS avg_total_revenue,
    ROUND(AVG(
        JULIANDAY(ce.churn_date) - JULIANDAY(a.signup_date)
    ), 0)                               AS avg_tenure_days,
    ROUND(AVG(ce.refund_amount_usd), 2) AS avg_refund,
    SUM(CASE WHEN ce.is_reactivation = 1 THEN 1 ELSE 0 END)
                                        AS reactivation_count
FROM churn_events ce
JOIN accounts a ON ce.account_id = a.account_id
GROUP BY ce.reason_code
HAVING COUNT(DISTINCT ce.account_id) >= 5
ORDER BY unique_players DESC;
"""

result_6 = query(sql_6)
print("Churn reasons with revenue and tenure context:")
result_6

Churn reasons with revenue and tenure context:


,reason_code,unique_players,avg_total_revenue,avg_tenure_days,avg_refund,reactivation_count
0,features,104,22628.07,178.0,16.72,13
1,support,95,22347.57,198.0,11.73,6
2,budget,94,21787.10,157.0,12.00,12
3,unknown,86,21298.45,160.0,18.34,9
4,pricing,86,21310.30,134.0,14.65,9
5,competitor,79,22385.47,204.0,13.08,12


---
## 7. Feature Usage Engagement Percentiles

**Business Question:** How does gameplay feature engagement distribute across players? Which features have the highest adoption rate?

**SQL Skills:** NTILE() for percentile bucketing, COUNT DISTINCT, feature adoption analysis

In [8]:
sql_7 = """
-- Feature adoption and engagement intensity
-- PySpark: feature_usage.groupBy('feature_name').agg(
--     F.countDistinct('subscription_id').alias('unique_users'),
--     F.sum('usage_count').alias('total_usage')
-- ).withColumn('adoption_pct', F.col('unique_users') / total_users)

WITH feature_stats AS (
    SELECT
        fu.feature_name,
        COUNT(DISTINCT s.account_id)         AS unique_players,
        SUM(fu.usage_count)                  AS total_usage,
        AVG(fu.usage_duration_secs)          AS avg_duration_secs,
        SUM(fu.error_count)                  AS total_errors
    FROM feature_usage fu
    JOIN subscriptions s ON fu.subscription_id = s.subscription_id
    GROUP BY fu.feature_name
),
total_players AS (
    SELECT COUNT(DISTINCT account_id) AS total FROM accounts
)
SELECT
    fs.feature_name,
    fs.unique_players,
    ROUND(100.0 * fs.unique_players / tp.total, 1)  AS adoption_rate_pct,
    fs.total_usage,
    ROUND(fs.avg_duration_secs, 0)                   AS avg_duration_secs,
    ROUND(1.0 * fs.total_errors / NULLIF(fs.total_usage, 0), 4) AS error_rate
FROM feature_stats fs
CROSS JOIN total_players tp
ORDER BY adoption_rate_pct DESC
LIMIT 15;
"""

result_7 = query(sql_7)
print("Top 15 features by player adoption rate:")
result_7

Top 15 features by player adoption rate:


,feature_name,unique_players,adoption_rate_pct,total_usage,avg_duration_secs,error_rate
0,feature_12,374,74.8,6534,3128.0,0.0549
1,feature_17,366,73.2,6458,2997.0,0.0571
2,feature_2,360,72.0,6525,3015.0,0.0615
3,feature_36,359,71.8,6389,2893.0,0.0581
4,feature_39,356,71.2,6430,3083.0,0.0555
5,feature_32,354,70.8,6686,3114.0,0.0534
6,feature_38,353,70.6,6478,3021.0,0.0550
7,feature_14,352,70.4,5880,2989.0,0.0522
8,feature_26,352,70.4,6470,3045.0,0.0645
9,feature_15,351,70.2,6621,3073.0,0.0527


---
## 8. Cumulative Revenue with Running SUM

**Business Question:** How does cumulative revenue grow over time for the business? Are there seasonal patterns or acceleration in monthly revenue?

**SQL Skills:** Running SUM() window function, date truncation, business-level aggregation

In [9]:
sql_8 = """
-- Cumulative monthly revenue with running total
-- PySpark: df.groupBy('month').agg(F.sum('mrr_amount'))
--          .withColumn('cumulative', F.sum('monthly_rev').over(Window.orderBy('month')))

WITH monthly AS (
    SELECT
        SUBSTR(start_date, 1, 7)     AS month,
        SUM(mrr_amount)              AS monthly_revenue,
        COUNT(DISTINCT account_id)   AS active_subscribers,
        COUNT(*)                     AS new_subscriptions
    FROM subscriptions
    GROUP BY SUBSTR(start_date, 1, 7)
)
SELECT
    month,
    ROUND(monthly_revenue, 2)        AS monthly_revenue,
    active_subscribers,
    new_subscriptions,
    ROUND(
        SUM(monthly_revenue) OVER (ORDER BY month ROWS UNBOUNDED PRECEDING),
        2
    )                                AS cumulative_revenue,
    ROUND(
        AVG(monthly_revenue) OVER (ORDER BY month ROWS BETWEEN 2 PRECEDING AND CURRENT ROW),
        2
    )                                AS rolling_3m_avg
FROM monthly
ORDER BY month;
"""

result_8 = query(sql_8)
print("Monthly revenue with cumulative total and 3-month rolling average:")
result_8

Monthly revenue with cumulative total and 3-month rolling average:


,month,monthly_revenue,active_subscribers,new_subscriptions,cumulative_revenue,rolling_3m_avg
0,2023-01,4684.0,2,3,4684.0,4684.00
1,2023-02,11079.0,9,11,15763.0,7881.50
2,2023-03,25885.0,16,17,41648.0,13882.67
3,2023-04,41788.0,25,31,83436.0,26250.67
4,2023-05,85919.0,26,29,169355.0,51197.33
5,2023-06,74987.0,36,46,244342.0,67564.67
6,2023-07,120422.0,45,59,364764.0,93776.00
7,2023-08,165621.0,63,82,530385.0,120343.33
8,2023-09,116222.0,51,62,646607.0,134088.33
9,2023-10,182518.0,65,84,829125.0,154787.00


---
## 9. Support SLA Compliance Analysis

**Business Question:** What percentage of support tickets are resolved within 24 hours? How does resolution speed vary by priority level and player tier?

**SQL Skills:** CASE WHEN bucketing, multi-level GROUP BY, SLA metric calculation

In [10]:
sql_9 = """
-- Support SLA compliance by priority and plan tier
-- PySpark: tickets.join(accounts, 'account_id')
--          .withColumn('sla_bucket', F.when(...).when(...).otherwise(...))
--          .groupBy('priority', 'plan_tier').agg(...)

SELECT
    t.priority,
    a.plan_tier,
    COUNT(*)                                                    AS total_tickets,
    ROUND(AVG(t.resolution_time_hours), 1)                      AS avg_resolution_hrs,
    ROUND(100.0 * SUM(CASE WHEN t.resolution_time_hours <= 24 THEN 1 ELSE 0 END)
          / COUNT(*), 1)                                        AS pct_resolved_24h,
    ROUND(100.0 * SUM(CASE WHEN t.resolution_time_hours <= 72 THEN 1 ELSE 0 END)
          / COUNT(*), 1)                                        AS pct_resolved_72h,
    ROUND(AVG(t.satisfaction_score), 2)                          AS avg_satisfaction,
    SUM(t.escalation_flag)                                       AS escalations
FROM support_tickets t
JOIN accounts a ON t.account_id = a.account_id
GROUP BY t.priority, a.plan_tier
ORDER BY t.priority, a.plan_tier;
"""

result_9 = query(sql_9)
print("Support SLA compliance by priority and tier:")
result_9

Support SLA compliance by priority and tier:


,priority,plan_tier,total_tickets,avg_resolution_hrs,pct_resolved_24h,pct_resolved_72h,avg_satisfaction,escalations
0,high,Basic,155,37.5,32.3,100.0,4.08,5
1,high,Enterprise,169,37.8,33.7,100.0,3.92,7
2,high,Pro,186,35.7,35.5,100.0,4.00,12
3,low,Basic,148,36.5,32.4,100.0,4.05,4
4,low,Enterprise,161,36.7,37.3,100.0,4.15,10
5,low,Pro,176,35.9,36.4,100.0,3.88,9
6,medium,Basic,149,34.0,40.3,100.0,4.00,8
7,medium,Enterprise,150,37.5,29.3,100.0,3.94,8
8,medium,Pro,192,35.3,34.9,100.0,3.89,9
9,urgent,Basic,178,34.4,39.3,100.0,4.04,6


---
## 10. First Purchase Funnel with FIRST_VALUE

**Business Question:** How quickly do players make their first paid subscription after signing up? How does conversion speed vary by referral source?

**SQL Skills:** FIRST_VALUE() / MIN() window function, date difference calculation, funnel analysis

In [11]:
sql_10 = """
-- Time-to-first-purchase by acquisition channel
-- PySpark: subs.filter(~F.col('is_trial'))
--          .groupBy('account_id').agg(F.min('start_date').alias('first_purchase'))

WITH first_paid AS (
    SELECT
        s.account_id,
        MIN(s.start_date) AS first_paid_date
    FROM subscriptions s
    WHERE s.is_trial = 0
    GROUP BY s.account_id
)
SELECT
    a.referral_source,
    COUNT(DISTINCT a.account_id)        AS total_players,
    COUNT(DISTINCT fp.account_id)       AS converted_players,
    ROUND(100.0 * COUNT(DISTINCT fp.account_id) / COUNT(DISTINCT a.account_id), 1)
                                        AS conversion_rate_pct,
    ROUND(AVG(
        JULIANDAY(fp.first_paid_date) - JULIANDAY(a.signup_date)
    ), 1)                               AS avg_days_to_first_purchase,
    ROUND(MIN(
        JULIANDAY(fp.first_paid_date) - JULIANDAY(a.signup_date)
    ), 0)                               AS fastest_conversion_days
FROM accounts a
LEFT JOIN first_paid fp ON a.account_id = fp.account_id
GROUP BY a.referral_source
ORDER BY conversion_rate_pct DESC;
"""

result_10 = query(sql_10)
print("Time-to-first-purchase funnel by channel:")
result_10

Time-to-first-purchase funnel by channel:


,referral_source,total_players,converted_players,conversion_rate_pct,avg_days_to_first_purchase,fastest_conversion_days
0,partner,89,89,100.0,38.7,0.0
1,other,103,103,100.0,47.5,0.0
2,organic,114,114,100.0,37.5,0.0
3,event,96,96,100.0,46.8,0.0
4,ads,98,98,100.0,37.1,0.0


---
## 11. Player LTV Segmentation with CASE WHEN Bucketing

**Business Question:** How should the UA team segment players for targeted campaigns? Create value tiers based on revenue and engagement.

**SQL Skills:** Nested CASE WHEN, multi-table JOIN, business-rule bucketing

In [12]:
sql_11 = """
-- Player value segmentation: Whale / Dolphin / Minnow / Free
-- PySpark: df.withColumn('value_segment',
--     F.when(F.col('total_mrr') > 10000, 'Whale')
--      .when(F.col('total_mrr') > 2000, 'Dolphin')
--      .when(F.col('total_mrr') > 0, 'Minnow')
--      .otherwise('Free'))

WITH player_metrics AS (
    SELECT
        a.account_id,
        a.plan_tier,
        a.churn_flag,
        COALESCE(SUM(s.mrr_amount), 0)          AS total_revenue,
        COUNT(DISTINCT s.subscription_id)        AS sub_count,
        JULIANDAY('2024-12-31') - JULIANDAY(a.signup_date) AS tenure_days
    FROM accounts a
    LEFT JOIN subscriptions s ON a.account_id = s.account_id
    GROUP BY a.account_id, a.plan_tier, a.churn_flag, a.signup_date
)
SELECT
    CASE
        WHEN total_revenue > 10000 THEN 'Whale'
        WHEN total_revenue > 2000  THEN 'Dolphin'
        WHEN total_revenue > 0     THEN 'Minnow'
        ELSE 'Free'
    END AS value_segment,
    COUNT(*)                                        AS player_count,
    ROUND(AVG(total_revenue), 2)                    AS avg_revenue,
    ROUND(SUM(total_revenue), 2)                    AS segment_total_revenue,
    ROUND(100.0 * SUM(CASE WHEN churn_flag = 1 THEN 1 ELSE 0 END) / COUNT(*), 1)
                                                    AS churn_rate_pct,
    ROUND(AVG(tenure_days), 0)                      AS avg_tenure_days
FROM player_metrics
GROUP BY value_segment
ORDER BY avg_revenue DESC;
"""

result_11 = query(sql_11)
print("Player value segmentation:")
result_11

Player value segmentation:


,value_segment,player_count,avg_revenue,segment_total_revenue,churn_rate_pct,avg_tenure_days
0,Whale,409,26401.09,10798046.0,21.8,338.0
1,Dolphin,83,6402.33,531393.0,24.1,350.0
2,Minnow,8,1163.50,9308.0,12.5,241.0


---
## 12. Subscription Upgrade/Downgrade Patterns

**Business Question:** How common are subscription tier changes? Do players who upgrade retain longer than those who downgrade?

**SQL Skills:** Self-referencing pattern analysis, conditional logic, business KPI calculation

In [13]:
sql_12 = """
-- Upgrade vs downgrade behaviour and retention impact
-- PySpark: subs.groupBy('account_id').agg(
--     F.sum('upgrade_flag').alias('upgrades'),
--     F.sum('downgrade_flag').alias('downgrades')
-- ).join(accounts, 'account_id')

WITH tier_changes AS (
    SELECT
        s.account_id,
        SUM(s.upgrade_flag)     AS total_upgrades,
        SUM(s.downgrade_flag)   AS total_downgrades,
        CASE
            WHEN SUM(s.upgrade_flag) > 0 AND SUM(s.downgrade_flag) = 0 THEN 'Upgrader'
            WHEN SUM(s.downgrade_flag) > 0 AND SUM(s.upgrade_flag) = 0 THEN 'Downgrader'
            WHEN SUM(s.upgrade_flag) > 0 AND SUM(s.downgrade_flag) > 0 THEN 'Volatile'
            ELSE 'Stable'
        END AS change_pattern
    FROM subscriptions s
    GROUP BY s.account_id
)
SELECT
    tc.change_pattern,
    COUNT(*)                                        AS player_count,
    ROUND(100.0 * SUM(CASE WHEN a.churn_flag = 1 THEN 1 ELSE 0 END) / COUNT(*), 1)
                                                    AS churn_rate_pct,
    ROUND(AVG(
        (SELECT SUM(s2.mrr_amount) FROM subscriptions s2 WHERE s2.account_id = tc.account_id)
    ), 2)                                           AS avg_total_revenue,
    ROUND(AVG(
        JULIANDAY('2024-12-31') - JULIANDAY(a.signup_date)
    ), 0)                                           AS avg_tenure_days
FROM tier_changes tc
JOIN accounts a ON tc.account_id = a.account_id
GROUP BY tc.change_pattern
ORDER BY avg_total_revenue DESC;
"""

result_12 = query(sql_12)
print("Subscription change patterns and retention impact:")
result_12

Subscription change patterns and retention impact:


,change_pattern,player_count,churn_rate_pct,avg_total_revenue,avg_tenure_days
0,Volatile,117,16.2,25217.37,335.0
1,Upgrader,190,23.7,23982.51,385.0
2,Downgrader,60,30.0,21944.57,314.0
3,Stable,133,21.1,18909.50,288.0


---
## PySpark Translation Guide

Each query above includes PySpark equivalents in comments. Below is a comprehensive reference for translating the key SQL patterns used in this notebook to PySpark.

### Pattern Mapping: SQL to PySpark

| SQL Pattern | PySpark Equivalent |
|---|---|
| `SELECT col1, col2 FROM table` | `df.select('col1', 'col2')` |
| `WHERE condition` | `df.filter(condition)` or `df.where(condition)` |
| `GROUP BY col` | `df.groupBy('col')` |
| `JOIN t2 ON t1.id = t2.id` | `df1.join(df2, 'id', 'inner')` |
| `LEFT JOIN` | `df1.join(df2, 'id', 'left')` |
| `CASE WHEN ... THEN ... ELSE ... END` | `F.when(cond, val).when(...).otherwise(val)` |
| `COALESCE(a, b)` | `F.coalesce(F.col('a'), F.col('b'))` |
| `RANK() OVER (PARTITION BY col ORDER BY col2)` | `F.rank().over(Window.partitionBy('col').orderBy('col2'))` |
| `LAG(col) OVER (...)` | `F.lag('col').over(window_spec)` |
| `SUM(col) OVER (ORDER BY col2 ROWS UNBOUNDED PRECEDING)` | `F.sum('col').over(Window.orderBy('col2').rowsBetween(Window.unboundedPreceding, 0))` |
| `COUNT(DISTINCT col)` | `F.countDistinct('col')` |
| `SUBSTR(date, 1, 7)` | `F.date_format('date', 'yyyy-MM')` |
| `JULIANDAY(d2) - JULIANDAY(d1)` | `F.datediff(F.col('d2'), F.col('d1'))` |
| CTE (`WITH ... AS (...)`) | Chain of `df = spark.sql(...)` or sequential DataFrame operations |

### Example: Full PySpark Translation of Query 4 (Cohort Retention)

```python
from pyspark.sql import functions as F, Window

# Load DataFrames
accounts = spark.read.csv('accounts.csv', header=True, inferSchema=True)
churn_events = spark.read.csv('churn_events.csv', header=True, inferSchema=True)

# Create cohort column
cohorts = accounts.withColumn(
    'signup_cohort', F.date_format('signup_date', 'yyyy-MM')
)

# Calculate tenure (days from signup to churn or analysis date)
last_churn = churn_events.groupBy('account_id').agg(
    F.max('churn_date').alias('last_churn_date')
)

cohorts = cohorts.join(last_churn, 'account_id', 'left')
cohorts = cohorts.withColumn(
    'tenure_days',
    F.when(
        F.col('churn_flag') == True,
        F.datediff(
            F.coalesce(F.col('last_churn_date'), F.lit('2024-12-31')),
            F.col('signup_date')
        )
    ).otherwise(
        F.datediff(F.lit('2024-12-31'), F.col('signup_date'))
    )
)

# Cohort retention metrics
result = cohorts.groupBy('signup_cohort').agg(
    F.count('*').alias('cohort_size'),
    F.round(
        100.0 * F.sum(F.when(F.col('tenure_days') >= 30, 1).otherwise(0)) / F.count('*'),
        1
    ).alias('retention_30d_pct'),
    F.round(
        100.0 * F.sum(F.when(F.col('tenure_days') >= 90, 1).otherwise(0)) / F.count('*'),
        1
    ).alias('retention_90d_pct'),
    F.round(
        100.0 * F.sum(F.when(F.col('tenure_days') >= 180, 1).otherwise(0)) / F.count('*'),
        1
    ).alias('retention_180d_pct'),
).orderBy('signup_cohort')

result.show()
```

### Production Deployment Note

In a production environment (e.g., Databricks on AWS), these queries would:
- Run on **Delta Lake** tables with ACID transactions and time travel
- Use **PySpark** for distributed computation on millions of player records
- Be scheduled via **Airflow/Databricks Jobs** for daily/hourly batch processing
- Feed into **Looker** dashboards via SQL views or materialized tables

The SQL patterns demonstrated here translate directly to production PySpark pipelines.

In [14]:
# Clean up
conn.close()
print("SQL analysis complete. 12 queries demonstrated.")

SQL analysis complete. 12 queries demonstrated.
